
Project : Pentaho Log Intelligence

Layer   : Silver

Notebook: 02_Silver_Transformation_Catalina

Version : 1.0

Description:
Loads raw Pentaho log files from Unity Catalog Volume

into the Silver Delta table.

Author: Ernesto Felipe Garay Cervantes


#### Recibimiento de Parametros 

In [0]:
import json

dbutils.widgets.text("archivos_nuevos","")

archivos_nuevos = json.loads(dbutils.widgets.get("archivos_nuevos"))

print("====ARCHIVOS RECIBIDOS===")
for archivo in archivos_nuevos:
    print(archivo)

#### Configuracion

In [0]:
from pyspark.sql.functions import (
    col,
    regexp_extract,
    current_timestamp
)

In [0]:
CATALOG = "pentaho_logs"

BRONZE_TABLE_CATALINA = "pentaho_logs.bronze.bronze_logs_catalina"

SILVER_TABLE_CATALINA= "pentaho_logs.silver.silver_logs_catalina"

#### Lectura  de tabla Bronze 

In [0]:
df_catalina_sl = spark.table(BRONZE_TABLE_CATALINA).filter(col("file_name").isin(archivos_nuevos))

#display(df_catalina_sl.limit(20))

In [0]:
df_catalina_sl.printSchema()

#### Enriquecimiento Data Frame

In [0]:
from pyspark.sql.functions import (col,regexp_extract,to_timestamp,to_date,date_format,try_to_date,when,trim)

df_silver_catalina = (df_catalina_sl.withColumn("mensaje", when(col("is_event_start") == True,trim(regexp_extract(col("descripcion"),r"^\d{2}-[A-Za-z]{3}-\d{4}\s+\d{2}:\d{2}:\d{2}\.\d{3}\s+(?:INFO|WARNING|SEVERE|DEBUG)\s+\[[^\]]+\]\s+\S+\s*(.*)$",1))).otherwise(trim(col("descripcion"))))
 .withColumn("fecha",date_format(try_to_date(col("fecha"), "dd-MMM-yyyy")," yyyy-MM-dd"))
                      )
#display(display(df_silver_catalina))



#### validación DATAFRAME

In [0]:
# Número de registros
print(f"Total de líneas: {df_silver_catalina.count():,}")

# Estructura
df_silver_catalina.printSchema()

In [0]:
from pyspark.sql.functions import col, sum, when

df_silver_catalina.select(
    sum(when(col("mensaje").isNull(), 1).otherwise(0)).alias("mensaje_evento_null"),
    sum(when(col("fecha").isNull(), 1).otherwise(0)).alias("fecha_null")
).show()


#### Creación Tabla Silver Carte 

In [0]:
SILVER_TABLE_CATALINA = "pentaho_logs.silver.silver_logs_catalina"
(
    df_silver_catalina.write
        .format("delta")
        .mode("append")
        .saveAsTable(SILVER_TABLE_CATALINA)
)

In [0]:
#display(spark.table(SILVER_TABLE_CATALINA).limit(20))